In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
import io
import sys
import gradio as gr
from IPython.display import Markdown,display

In [ ]:
load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key=os.getenv('OPENROUTER_API_KEY')

if openrouter_api_key:
  print(f"Openrouter api key is present {openrouter_api_key[:8]}")
else:
  print("Openrouter api key not set")

if groq_api_key:
  print(f"Grok Api Key present {groq_api_key[:8]}")
else:
  print(f"No groq api key present")

if google_api_key:
  print(f"Google api key present {google_api_key[:8]}")
else:
  print(f"Google api key not present")


In [ ]:
# We will use the open source models cause we are still poo bitch ->
openai = OpenAI()
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

In [ ]:
gemini = OpenAI(api_key=google_api_key,base_url=gemini_url)
groq = OpenAI(api_key=groq_api_key,base_url=groq_url)
ollama = OpenAI(api_key="ollama",base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key,base_url=openrouter_url)

In [ ]:
models = [
    "gemma4:cloud",
    "llama3.2:latest",
    "gpt-oss:20b-cloud",
]

clients = {
    model: ollama
    for model in models
}

In [ ]:
from system_info import retrieve_system_info
system_info = retrieve_system_info()
system_info

In [ ]:
compile_command = ["clang++", "-std=c++17", "-Ofast", "-mcpu=native", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]


# And now, on with the main task ->

In [ ]:
system_prompt = """
Your task is to convert Python code into high performance C++ code. Respond only with C++ code. Do not provide any axplanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
  return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```Python
{python}
```
"""

In [ ]:
def messages_for(python):
  return[
    {"role":"system","content":system_prompt},
    {"role":"user","content":user_prompt_for(python)}
  ]

In [ ]:
def write_output(cpp):
  with open("main.cpp","w") as f:
    f.write(cpp)

In [ ]:
def port(model,python):
  client = clients[model]
  response = client.chat.completions.create(model=model,messages=messages_for(python))
  reply = response.choices[0].message.content
  reply = reply.replace('```cpp','').replace('```','')
  write_output(reply)
  return reply

In [ ]:
pi = """
import time

def calculate(iterations,param1,param2):
  result = 1.0
  for i in range(1, iterations+1):
    j = i * param1 - param2
    result -= (1/j)
    j = i * param1 + param2
    result += (1/j)
  return result

start_time = time.time()
result = calculate(200_000_000,4,1)*4
end_time = time.time()
print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [ ]:
def run_python(code):
  globals_dict = {"__builtin__":__builtins__}

  buffer = io.StringIO()
  old_stdout = sys.stdout
  sys.stdout = buffer

  try:
    exec(code,globals_dict)
    output = buffer.getvalue()
  except Exception as e:
    output = f"error: {e}"
  finally:
    sys.stdout = old_stdout
  return output

In [ ]:
def compile_and_run():
  try:
    subprocess.run(compile_command,check=True, text=True,capture_output=True)
    print(subprocess.run(run_command,check=True,text=True,capture_output=True).stdout)
    print(subprocess.run(run_command,check=True,text=True,capture_output=True).stdout)
    print(subprocess.run(run_command,check=True,text=True,capture_output=True).stdout)
  except subprocess.CalledProcessError as e:
    print(f"An error occured:\n{e.stderr}")

In [ ]:
with gr.Blocks() as ui:
  with gr.Row():
    python = gr.Textbox(label="Python Code: ",lines=28,value=pi)
    cpp = gr.Textbox(label="C++ Code: ",lines=28)
    with gr.Row():
      model=gr.Dropdown(models,label="Select Model",value=models[0])
      convert = gr.Button("Convert code")

    convert.click(port,inputs=[model,python],outputs=[cpp])
ui.launch(inbrowser=True)

In [ ]:
compile_and_run()

## This is the result of the model gemma4:cloud
Result: 3.141592656089
Execution Time: 0.187785 seconds

Result: 3.141592656089
Execution Time: 0.185807 seconds

Result: 3.141592656089
Execution Time: 0.191010 seconds

## This is the result of the model llama3.2:latest
Result: 3.141592656089
Execution Time: 0.577656 seconds

Result: 3.141592656089
Execution Time: 0.272121 seconds

Result: 3.141592656089
Execution Time: 0.204086 seconds

## This is the result of the model gpt-oss:20b-cloud
Result: 3.141592656090
Execution Time: 0.327761 seconds

Result: 3.141592656090
Execution Time: 0.451839 seconds

Result: 3.141592656090
Execution Time: 0.330081 seconds